# Reflection and Blogpost Writing

## Setup

In [ ]:
%pip install -q autogen-agentchat autogen-ext[openai] python-dotenv openai

In [1]:
from dotenv import load_dotenv
import os

load_dotenv("untitled.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found — check untitled.env")

# Shared model config used by all agents
MODEL = "gpt-3.5-turbo"

In [2]:
import autogen_agentchat
print("autogen-agentchat version:", autogen_agentchat.__version__)

autogen-agentchat version: 0.7.5


## The Task

In [3]:
task = (
    "Write a concise but engaging blogpost about Hope AI. "
    "Make sure the blogpost is within 100 words."
)

## Create Agents

In [4]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
import asyncio

# Shared OpenAI client
openai_client = OpenAIChatCompletionClient(
    model=MODEL,
    api_key=OPENAI_API_KEY,
)

In [5]:
writer = AssistantAgent(
    name="Writer",
    model_client=openai_client,
    system_message=(
        "You are a writer. You write engaging and concise blog posts "
        "(with title) on given topics. You must polish your writing "
        "based on feedback you receive and provide a refined version. "
        "Only return the final blog post."
    ),
)

## Quick Writer Test (no reflection yet)

In [6]:
# Run a single-agent task to verify the writer works
async def test_writer():
    termination = MaxMessageTermination(2)
    team = RoundRobinGroupChat([writer], termination_condition=termination)
    await Console(team.run_stream(task=task))
    await team.reset()

await test_writer()

---------- TextMessage (user) ----------
Write a concise but engaging blogpost about Hope AI. Make sure the blogpost is within 100 words.


Error processing publish message for Writer_84aff1bf-d0b2-492f-a4aa-8338aaf18648/84aff1bf-d0b2-492f-a4aa-8338aaf18648
Traceback (most recent call last):
  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib

RuntimeError: RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Traceback:
Traceback (most recent call last):

  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_agentchat\teams\_group_chat\_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):
    ...<4 lines>...
            await self._log_message(msg)

  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py", line 953, in on_messages_stream
    async for inference_output in self._call_llm(
    ...<15 lines>...
            yield inference_output

  File "C:\U

## Adding Reflection

Create a critic agent to reflect on the work of the writer agent.

In [7]:
critic = AssistantAgent(
    name="Critic",
    model_client=openai_client,
    system_message=(
        "You are a critic. You review the work of the writer and provide "
        "constructive feedback to help improve the quality of the content. "
        "When you are satisfied with the final version, respond with TERMINATE."
    ),
)

In [8]:
async def run_writer_critic():
    termination = (
        TextMentionTermination("TERMINATE") | MaxMessageTermination(6)
    )
    team = RoundRobinGroupChat(
        [writer, critic],
        termination_condition=termination,
    )
    result = await Console(team.run_stream(task=task))
    await team.reset()
    return result

result = await run_writer_critic()

---------- TextMessage (user) ----------
Write a concise but engaging blogpost about Hope AI. Make sure the blogpost is within 100 words.


Error processing publish message for Writer_6ff249dc-1c5b-4082-9b0d-5a8a535909a1/6ff249dc-1c5b-4082-9b0d-5a8a535909a1
Traceback (most recent call last):
  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib

RuntimeError: RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Traceback:
Traceback (most recent call last):

  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_agentchat\teams\_group_chat\_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):
    ...<4 lines>...
            await self._log_message(msg)

  File "C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py", line 953, in on_messages_stream
    async for inference_output in self._call_llm(
    ...<15 lines>...
            yield inference_output

  File "C:\U

## Nested Chat — Specialist Reviewers

In [9]:
SEO_reviewer = AssistantAgent(
    name="SEO_Reviewer",
    model_client=openai_client,
    system_message=(
        "You are an SEO reviewer, known for your ability to optimize "
        "content for search engines, ensuring that it ranks well and "
        "attracts organic traffic. Make sure your suggestion is concise "
        "(within 3 bullet points), concrete and to the point. "
        "Begin the review by stating your role. "
        "When done, respond with TERMINATE."
    ),
)

legal_reviewer = AssistantAgent(
    name="Legal_Reviewer",
    model_client=openai_client,
    system_message=(
        "You are a legal reviewer, known for your ability to ensure that "
        "content is legally compliant and free from any potential legal "
        "issues. Make sure your suggestion is concise (within 3 bullet "
        "points), concrete and to the point. "
        "Begin the review by stating your role. "
        "When done, respond with TERMINATE."
    ),
)

ethics_reviewer = AssistantAgent(
    name="Ethics_Reviewer",
    model_client=openai_client,
    system_message=(
        "You are an ethics reviewer, known for your ability to ensure "
        "that content is ethically sound and free from any potential "
        "ethical issues. Make sure your suggestion is concise (within 3 "
        "bullet points), concrete and to the point. "
        "Begin the review by stating your role. "
        "When done, respond with TERMINATE."
    ),
)

meta_reviewer = AssistantAgent(
    name="Meta_Reviewer",
    model_client=openai_client,
    system_message=(
        "You are a meta reviewer. You aggregate and review the work of "
        "other reviewers and give a final suggestion on the content. "
        "When done, respond with TERMINATE."
    ),
)

## Orchestrate the Full Nested Review Pipeline

In [10]:
async def run_nested_review(draft: str):
    """Run SEO, Legal, Ethics reviewers in sequence, then Meta aggregates."""
    review_prompt = f"Review the following blog post draft:\n\n{draft}"
    termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(3)

    reviews = {}

    for reviewer in [SEO_reviewer, legal_reviewer, ethics_reviewer]:
        team = RoundRobinGroupChat([reviewer], termination_condition=termination)
        res = await Console(team.run_stream(task=review_prompt))
        await team.reset()
        # Grab the last assistant message as the review
        reviews[reviewer.name] = res.messages[-1].content

    # Meta reviewer aggregates all specialist feedback
    meta_prompt = (
        "Aggregate the following specialist reviews and provide a final "
        "suggestion on the writing.\n\n"
        + "\n\n".join(f"{k}:\n{v}" for k, v in reviews.items())
    )
    meta_termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(3)
    meta_team = RoundRobinGroupChat([meta_reviewer], termination_condition=meta_termination)
    meta_result = await Console(meta_team.run_stream(task=meta_prompt))
    await meta_team.reset()
    return meta_result

# Get the draft from the writer-critic run
draft = result.messages[-1].content
print("\n--- Draft being reviewed ---\n", draft)

NameError: name 'result' is not defined

**Note:** You might get a slightly different response each run. Feel free to change the task above.

In [ ]:
final = await run_nested_review(draft)

## Final Summary

In [ ]:
print("\n=== FINAL META-REVIEWER SUGGESTION ===")
print(final.messages[-1].content)